<h1><center>
    Лекція 10.
    <div>Градієнтний бустинг</div>
</center></h1>

Наразі ми вже розглянули 9 тем: від дослідницького аналізу даних до аналізу часових рядів у Python. Тепер звернемося до одного з найпопулярніших і найпрактичніших алгоритмів машинного навчання, а саме градієнтного бустингу.

<img src='https://habrastorage.org/web/4a9/edb/082/4a9edb082408442ea47a12b75f19d122.jpg' align='center' width=50%>

###  Зміст

1. [Вступ і історія бустингу](#1.--Introduction-and-history-of-boosting)
   - [Історія Gradient Boosting Machine](#History-of-GBM)
1. [Алгоритм Gradient Boosting Machine (GBM)](#2.-GBM-algorithm)
   - [Постановка задачі ML](#ML-problem-statement)
   - [Функціональний градієнтний спуск](#Functional-gradient-descent)
   - [Класичний алгоритм GBM за Фрідманом](#Friedman's-classic-GBM-algorithm)
   - [Покроковий приклад роботи алгоритму GBM](#Step-By-Step-example:-How-GBM-Works)
1. [Функції втрат](#3.-Loss-functions)
   - [Функції втрат для регресії](#Regression-loss-functions)
   - [Функції втрат для класифікації](#Classification-loss-functions)
   - [Ваги](#Weights)
1. [Висновки](#4.-Conclusion)
1. [Корисні ресурси](#5-Useful-resources)

<a id="1.--Introduction-and-history-of-boosting"></a>

# 1. Вступ і історія бустингу

Майже кожен, хто працює в машинному навчанні, чув про градієнтний бустинг. Багато фахівців із науки про дані включають цей алгоритм до свого інструментарію через стабільно високі результати, які він дає на довільних, наперед невідомих задачах.

Крім того, [XGBoost](https://en.wikipedia.org/wiki/XGBoost) часто вважають стандартним рецептом для [перемоги](https://github.com/dmlc/xgboost/blob/master/demo/README.md#usecases) в [ML-змаганнях](http://blog.kaggle.com/tag/xgboost/). Його популярність настільки велика, що ідея стекінгу з кількох XGBoost-моделей уже стала мемом. Також бустинг є важливим компонентом [багатьох рекомендаторних систем](https://en.wikipedia.org/wiki/Learning_to_rank#Practical_usage_by_search_engines); інколи його навіть сприймають як окремий [бренд](https://yandex.com/company/technologies/matrixnet/).
Розгляньмо історію виникнення та розвитку бустингу.

Бустинг виник із [такого запитання](http://www.cis.upenn.edu/~mkearns/papers/boostnote.pdf): чи можна отримати одну сильну модель із великої кількості відносно слабких і простих моделей? Під «слабкими моделями» тут маються на увазі не примітивні базові алгоритми на кшталт дерев рішень, а моделі з низькою точністю, тобто такі, що працюють лише трохи краще за випадкове вгадування.

[Позитивну математичну відповідь](http://www.cs.princeton.edu/~schapire/papers/strengthofweak.pdf) на це запитання було отримано, але знадобилося ще кілька років, щоб побудувати повноцінні алгоритми на основі цієї ідеї, наприклад [AdaBoost](https://en.wikipedia.org/wiki/AdaBoost). Ці алгоритми використовують жадібний підхід: спочатку вони будують лінійну комбінацію простих моделей (базових алгоритмів), переважуючи вхідні дані. Далі модель, зазвичай дерево рішень, навчається на об’єктах, які раніше було класифіковано неправильно, і саме ці об’єкти отримують більші ваги.

У багатьох курсах із машинного навчання вивчають AdaBoost як попередника GBM (Gradient Boosting Machine). Однак після зближення ідей AdaBoost і GBM стало зрозуміло, що AdaBoost є лише окремим випадком GBM. Сам алгоритм має дуже наочну візуальну інтерпретацію та зрозумілу інтуїцію щодо визначення ваг.

Розгляньмо таку іграшкову задачу класифікації, у якій на кожній ітерації AdaBoost ми будемо розбивати дані між деревами глибини 1 (так званими пеньками, або stumps). Для перших двох ітерацій маємо таку картину:

<img src='https://habrastorage.org/web/d28/78f/7ba/d2878f7bad0340fc8002e5ba6d0879a5.jpg' align='center' width=65%>

Розмір точки відповідає її вазі, яку було призначено за неправильне передбачення. На кожній ітерації видно, що ці ваги зростають, адже пеньки не можуть впоратися з цією задачею. Водночас, якщо виконати зважене голосування для цих пеньків, отримаємо правильну класифікацію:

<img src='https://habrastorage.org/web/b2b/029/d89/b2b029d898f64bbbb158e15d29595969.png' align='center' width=70%>

Псевдокод:

- Ініціалізуйте ваги об’єктів: $w_i^{(0)} = \frac{1}{l}, \quad i = 1, \dots, l$.
- Для всіх $t = 1, \dots, T$:
    * навчіть базовий алгоритм $b_t$, нехай $\epsilon_t$ – його похибка на навчанні;
    * обчисліть $\alpha_t = \frac{1}{2}\ln\frac{1 - \epsilon_t}{\epsilon_t}$;
    * оновіть ваги об’єктів: $w_i^{(t)} = w_i^{(t-1)} e^{-\alpha_t y_i b_t(x_i)}, \quad i = 1, \dots, l$;
    * нормалізуйте ваги об’єктів: $w_0^{(t)} = \sum_{j = 1}^{l} w_j^{(t)}, \quad w_i^{(t)} = \frac{w_i^{(t)}}{w_0^{(t)}}, \quad i = 1, \dots, l$.
- Поверніть $\sum_{t=1}^{T} \alpha_t b_t$.

[Тут](https://www.youtube.com/watch?v=k4G2VCuOMMg) наведено докладніший приклад AdaBoost, у якому в міру ітерацій можна побачити, як зростають ваги, особливо на межі між класами.

AdaBoost працює добре, однак [відсутність](https://www.cs.princeton.edu/courses/archive/spring07/cos424/papers/boosting-survey.pdf) чіткого пояснення того, чому алгоритм є успішним, породжувала сумніви. Одні вважали його надалгоритмом, універсальним засобом на всі випадки, інші ж ставилися скептично і припускали, що AdaBoost просто перенавчається.

Проблема перенавчання справді існувала, особливо коли в даних були сильні викиди. У таких задачах AdaBoost виявлявся нестабільним. На щастя, кілька професорів зі статистичного факультету Стенфорда, які раніше створили Lasso, Elastic Net і Random Forest, почали досліджувати цей алгоритм.

У 1999 році Джером Фрідман запропонував узагальнення розвитку алгоритмів бустингу – Gradient Boosting (Machine), також відомий як GBM.

У своїй праці [Greedy Function Approximation: A Gradient Boosting Machine](https://jerryfriedman.su.domains/ftp/trebst.pdf) Фрідман заклав статистичний фундамент для багатьох алгоритмів, сформулювавши загальний підхід до бустингу як методу оптимізації у функціональному просторі.

[CART](https://en.wikipedia.org/wiki/Decision_tree_learning#cite_note-bfos-7), bootstrap та багато інших алгоритмів виникли саме на статистичному факультеті Стенфорда. Завдяки цьому цей підрозділ міцно закріпив свої імена в майбутніх підручниках. Ці алгоритми надзвичайно практичні, а деякі новіші роботи досі ще не набули широкого поширення. Наприклад, зверніть увагу на [glinternet](https://arxiv.org/abs/1308.2719).

Збереглося не так багато відеозаписів із Фрідманом. Проте є дуже цікаве [інтерв’ю](https://www.youtube.com/watch?v=8hupHmBVvb0), у якому він розповідає про створення CART і про те, як вони розв’язували статистичні задачі, що багато в чому нагадує сучасний аналіз даних і науку про дані понад 40 років потому.

Також існує чудова [лекція](https://www.youtube.com/watch?v=zBk3PK3g-Fc) Хасті – ретроспективний погляд на аналіз даних від одного зі співавторів методів, якими ми користуємося щодня.

Загалом відбувся перехід від суто інженерних і алгоритмічних досліджень до повноцінного підходу до побудови й вивчення алгоритмів. З математичного погляду зміни не є радикальними: ми, як і раніше, додаємо слабкі алгоритми (тобто підсилюємо їх) і нарощуємо ансамбль поступовими поліпшеннями на тих частинах даних, де модель працювала неточно.

Однак тепер наступна проста модель будується не просто на переважених об’єктах, а покращує наближення градієнта глобальної цільової функції. Саме ця ідея суттєво розширює простір для розвитку наших алгоритмів та їхніх модифікацій.

<img src='https://habrastorage.org/webt/h2/v4/k9/h2v4k9r-4yn4jwvwz99fbss4ghi.png' align='center' width=100%/>

<a id="History-of-GBM"></a>

## Історія GBM

Минуло понад 10 років після появи GBM, перш ніж він став невід’ємною частиною інструментарію науки про дані.

GBM було розширено для застосування до різних статистичних задач: GLMboost і GAMboost для підсилення вже наявних GAM-моделей, CoxBoost для кривих виживаності, RankBoost і LambdaMART для ранжування.

Також з’явилося багато реалізацій GBM під різними назвами й на різних платформах: Stochastic GBM, GBDT (градієнтно-бустингові дерева рішень), GBRT (градієнтно-бустингові дерева регресії), MART (багатоадитивні дерева регресії) та інші. Крім того, ML-спільнота була доволі сегментованою та роз’єднаною, через що було складно простежити, наскільки широко вже поширився бустинг.

У той самий час бустинг активно застосовували в задачах ранжування пошуку. Цю задачу переписали в термінах функції втрат, яка штрафує помилки у вихідному порядку об’єктів, тому її стало зручно безпосередньо вбудовувати в GBM. Однією з перших компаній, що впровадили бустинг для ранжування, була AltaVista. Згодом ці ідеї поширилися на Yahoo, Bing та інші системи.

Після цього бустинг став одним із головних алгоритмів, який використовували не лише в дослідженнях, а й у ключових промислових технологіях.

<img src='https://habrastorage.org/web/48a/ea4/fff/48aea4fffdbe4e5f9205ba81110e6061.jpg' align='center' width=40%> 

ML-змагання, особливо Kaggle, відіграли ключову роль у популяризації бустингу. Дослідники отримали спільну платформу, де могли змагатися на різних задачах науки про дані за участю великої кількості учасників з усього світу. Завдяки Kaggle стало можливим перевіряти нові алгоритми на реальних даних, даючи їм можливість проявити себе, а також відкрито порівнювати результати моделей на конкурсних наборах даних.

Саме це і сталося з бустингом, коли його почали застосовувати на [Kaggle](http://blog.kaggle.com/2011/12/21/score-xavier-conort-on-coming-second-in-give-me-some-credit/) (достатньо подивитися інтерв’ю з призерами Kaggle, починаючи з 2011 року, де більшість із них використовували бустинг). Бібліотека [XGBoost](https://github.com/dmlc/xgboost) швидко набула популярності після своєї появи. XGBoost не є принципово новим або унікальним алгоритмом; це лише надзвичайно ефективна реалізація класичного GBM із додатковими евристиками.

Цей алгоритм пройшов типовий для сучасних ML-методів шлях: від математичної задачі та алгоритмічної майстерності до успішних практичних застосувань і масового впровадження через роки після першої появи.

<a id="2.-GBM-algorithm"></a>

# 2. Алгоритм Gradient Boosting Machine (GBM)

<a id="ML-problem-statement"></a>

### Постановка задачі ML

Ми розв’язуватимемо задачу апроксимації функції в загальній постановці навчання з учителем. Маємо набір ознак $x$ і цільові змінні $y, \left\{ (x_i, y_i) \right\}_{i=1, \ldots,n}$, за допомогою яких потрібно відновити залежність $y = f(x)$. Відновлюємо цю залежність, наближаючи $\hat{f}(x)$ і визначаючи, яке наближення є кращим за функцією втрат $L(y,f)$, яку ми прагнемо мінімізувати:

$$
\begin{aligned}
y &\approx \hat{f}(x), \\
\hat{f}(x) &= \arg\min_{f(x)} L\bigl(y, f(x)\bigr).
\end{aligned}
$$

<img src='https://raw.githubusercontent.com/radiukpavlo/intelligent-data-analysis/refs/heads/main/03_img/10_1_help_with_func.png'  align='center' width=150%>

На цьому етапі ми не робимо жодних припущень щодо типу залежності $f(x)$, моделі нашого наближення $\hat{f}(x)$ чи розподілу цільової змінної $y$. Ми лише очікуємо, що функція$L(y,f)$є диференційовною. Наш підхід є дуже загальним: ми визначаємо $\hat {f}(x)$ через мінімізацію втрат:

$$
\hat{f}(x) = \arg\min_{f(x)} \mathbb{E}_{x,y}\bigl[L\bigl(y, f(x)\bigr)\bigr].
$$

На жаль, кількість функцій $f(x)$ не просто велика – відповідний функціональний простір є нескінченновимірним. Саме тому доцільно обмежити простір пошуку певною сім’єю функцій $f(x, \theta), \theta \in \mathbb{R}^d$. Це суттєво спрощує цільову задачу, адже тепер ми отримуємо розв’язну оптимізацію за параметрами, $\hat{f}(x) = f(x, \hat{\theta}):$

$$
\hat{\theta} = \arg\min_{\theta} \mathbb{E}_{x,y}\bigl[L\bigl(y, f(x, \theta)\bigr)\bigr].
$$

Прості аналітичні розв’язки для знаходження оптимальних параметрів $\hat{\theta}$ часто відсутні, тому параметри зазвичай наближають ітеративно. Спочатку запишемо емпіричну функцію втрат $L_{\theta}(\hat{\theta})$, яка дасть змогу оцінювати параметри на наших даних. Додатково подамо наше наближення $\hat{\theta}$ за $M$ ітерацій у вигляді суми:

$$
\begin{aligned}
\hat{\theta} &= \sum_{i=1}^{M} \hat{\theta}_i, \\
L_{\theta}(\hat{\theta}) &= \sum_{i=1}^{N} L\bigl(y_i, f(x_i, \hat{\theta})\bigr).
\end{aligned}
$$

Після цього залишається лише знайти придатний ітеративний алгоритм для мінімізації $L_{\theta}(\hat{\theta})$. Найпростішим і найуживанішим варіантом є градієнтний спуск.

Позначимо градієнт як $\nabla L_{\theta}(\hat{\theta})$ і додаватимемо до нього наші ітеративні прирости $\hat{\theta}_i$ (оскільки ми мінімізуємо втрати, використовується знак мінус).

Останній крок полягає в ініціалізації першого наближення $\hat{\theta}_0$ і виборі кількості ітерацій $M$. Розгляньмо кроки цього неефективного, але наочного алгоритму наближення $\hat{\theta}$:

1. Визначте початкове наближення параметрів: $\hat{\theta} = \hat{\theta}_0$.
2. Для кожної ітерації $t = 1, \dots, M$ повторюйте:
    1. Обчисліть градієнт функції втрат для поточного наближення $\hat{\theta}$:
    $$
    \nabla L_{\theta}(\hat{\theta}) = \left[\frac{\partial L\bigl(y, f(x, \theta)\bigr)}{\partial \theta}\right]_{\theta = \hat{\theta}}.
    $$
    2. Визначте поточний ітеративний приріст:
    $$
    \hat{\theta}_t \leftarrow -\nabla L_{\theta}(\hat{\theta}).
    $$
    3. Оновіть наближення параметрів:
    $$
    \hat{\theta} \leftarrow \hat{\theta} + \hat{\theta}_t = \sum_{i=0}^{t} \hat{\theta}_i.
    $$
3. Збережіть результат наближення:
$$
\hat{\theta} = \sum_{i=0}^{M} \hat{\theta}_i.
$$
4. Використайте знайдену функцію $\hat{f}(x) = f(x, \hat{\theta})$.

<img src='https://habrastorage.org/web/2b5/5d6/90d/2b55d690d99e4ec0976b360aae6ce4df.jpg' align='center' width=150%>

<a id="Functional-gradient-descent"></a>

### Функціональний градієнтний спуск

Уявімо на мить, що ми можемо виконувати оптимізацію у просторі функцій і ітеративно шукати наближення $\hat{f}(x)$ як самі функції. Подамо наше наближення як суму послідовних покращень, кожне з яких також є функцією. Для зручності почнемо сумування з початкового наближення $\hat{f}_0(x):$

$$
\hat{f}(x) = \sum_{i=0}^{M} \hat{f}_i(x).
$$

Поки що нічого не сталося; ми лише вирішили шукати наближення $\hat{f}(x)$ не як велику модель із великою кількістю параметрів (наприклад, нейронну мережу), а як суму функцій, ніби рухаючись у функціональному просторі.

Щоб реалізувати цю ідею, потрібно обмежити пошук деякою сім’єю функцій $\hat{f}(x) = h(x, \theta)$. Тут виникає кілька питань: по-перше, сума моделей може бути складнішою за будь-яку окрему модель із цієї сім’ї; по-друге, загальна цільова задача все одно лишається у функціональному просторі. Зауважимо, що на кожному кроці нам доведеться вибирати оптимальний коефіцієнт $\rho \in \mathbb{R}$. Для кроку $t$ задача має такий вигляд:

$$
\begin{aligned}
\hat{f}(x) &= \sum_{i=0}^{t-1} \hat{f}_i(x), \\
(\rho_t, \theta_t) &= \arg\min_{\rho, \theta} \mathbb{E}_{x,y}\bigl[L\bigl(y, \hat{f}(x) + \rho \, h(x, \theta)\bigr)\bigr], \\
\hat{f}_t(x) &= \rho_t \, h(x, \theta_t).
\end{aligned}
$$

Саме тут і відбувається головне. Ми сформулювали всі наші цілі в загальному вигляді, наче могли б навчати будь-який тип моделі $h(x, \theta)$ для будь-якої функції втрат $L(y, f(x, \theta))$. На практиці це надзвичайно складно, але, на щастя, існує простий спосіб розв’язати цю задачу.

Знаючи вираз для градієнта функції втрат, ми можемо обчислити його значення на наших даних. Тому навчатимемо моделі так, щоб їхні передбачення були якомога більш узгодженими з цим градієнтом (зі знаком мінус).

Інакше кажучи, ми використовуватимемо метод найменших квадратів, щоб скоригувати передбачення цими залишками. Для задач класифікації, регресії та ранжування мінімізуватимемо квадрат різниці між псевдозалишками$r$і нашими передбаченнями. Для кроку$t$остаточна задача має такий вигляд:

$$
\begin{aligned}
\hat{f}(x) &= \sum_{i=0}^{t-1} \hat{f}_i(x), \\
r_{it} &= -\left[\frac{\partial L\bigl(y_i, f(x_i)\bigr)}{\partial f(x_i)}\right]_{f(x)=\hat{f}(x)}, \quad \text{для } i = 1, \ldots, n, \\
\theta_t &= \arg\min_{\theta} \sum_{i=1}^{n} \bigl(r_{it} - h(x_i, \theta)\bigr)^2, \\
\rho_t &= \arg\min_{\rho} \sum_{i=1}^{n} L\bigl(y_i, \hat{f}(x_i) + \rho \, h(x_i, \theta_t)\bigr).
\end{aligned}
$$

<img src='https://raw.githubusercontent.com/radiukpavlo/intelligent-data-analysis/refs/heads/main/03_img/10_2_regression_for_everybody.jpg'   align='center' width=100%>

<a id="Friedman's-classic-GBM-algorithm"></a>

### Класичний алгоритм GBM за Фрідманом

Тепер можемо сформулювати класичний алгоритм GBM, запропонований Джеромом Фрідманом у 1999 році. Це алгоритм навчання з учителем, який має такі складові:

- набір даних $\left\{ (x_i, y_i) \right\}_{i=1, \ldots,n}$;
- кількість ітерацій $M$;
- вибір функції втрат $L(y, f)$ з визначеним градієнтом;
- вибір сім’ї функцій базових алгоритмів $h(x, \theta)$ разом із процедурою навчання;
- додаткові гіперпараметри $h(x, \theta)$ (наприклад, глибина дерева рішень).

Залишається лише початкове наближення $f_0(x)$. Для простоти як початкове наближення використовують сталу величину $\gamma$. І цю сталу, і оптимальний коефіцієнт $\rho$ визначають за допомогою бінарного пошуку або іншого алгоритму line search на початковій функції втрат (а не на її градієнті). Отже, наш алгоритм GBM можна описати так:

- Ініціалізуйте GBM сталою величиною:
$$
\hat{f}(x) = \hat{f}_0 = \gamma, \qquad \gamma \in \mathbb{R}.
$$
$$
\hat{f}_0 = \arg\min_{\gamma} \sum_{i=1}^{n} L(y_i, \gamma).
$$
- Для кожної ітерації $t = 1, \dots, M$ повторюйте:
    1. Обчисліть псевдозалишки $r_t$:
    $$
    r_{it} = -\left[\frac{\partial L\bigl(y_i, f(x_i)\bigr)}{\partial f(x_i)}\right]_{f(x)=\hat{f}(x)}, \quad \text{для } i = 1, \ldots, n.
    $$

    2. Побудуйте новий базовий алгоритм $h_t(x)$ як регресію на псевдозалишках $\{(x_i, r_{it})\}_{i=1}^{n}.$

- Знайдіть оптимальний коефіцієнт $\rho_t$ для $h_t(x)$ відносно початкової функції втрат:
$$
\rho_t = \arg\min_{\rho} \sum_{i=1}^{n} L\bigl(y_i, \hat{f}(x_i) + \rho \, h(x_i, \theta)\bigr).
$$
- Збережіть $\hat{f}_t(x) = \rho_t \, h_t(x)$.
- Оновіть поточне наближення:
    $$\hat{f}(x) \leftarrow \hat{f}(x) + \hat{f}_t(x) = \sum_{i=0}^{t} \hat{f}_i(x).$$

    1. Сформуйте фінальну модель GBM:
    $$\hat{f}(x) = \sum_{i=0}^{M} \hat{f}_i(x).$$
    2. Підкоріть Kaggle і весь інший світ.

<a id="Step-By-Step-example:-How-GBM-Works"></a>

### Покроковий приклад: як працює GBM

Розгляньмо приклад того, як працює GBM. У цій іграшковій задачі ми відновлюватимемо зашумлену функцію $y = \cos(x) + \epsilon, \epsilon \sim \mathcal{N}(0, \frac{1}{5}), x \in [-5,5]$.

<img src='https://habrastorage.org/web/9fe/04d/7ba/9fe04d7ba5a645d49fc6aa3e875c8c41.jpg' align='center' width='100%'>

Це задача регресії з дійсною цільовою змінною, тому ми виберемо функцію втрат середньоквадратичної помилки. Згенеруємо 300 пар спостережень і апроксимуватимемо їх деревами рішень глибини 2. Зберімо все, що нам потрібно для використання GBM:

- іграшкові дані$\left\{ (x_i, y_i) \right\}_{i=1, \ldots,300}$;
- кількість ітерацій$M = 3$;
- функція втрат середньоквадратичної помилки$L(y, f) = (y-f)^2$;
- градієнт втрат$L(y, f) = L_2$ – це просто залишки$r = (y - f)$;
- дерева рішень як базові алгоритми$h(x)$;
- гіперпараметри дерев рішень: глибина дерева дорівнює 2.

Для середньоквадратичної помилки і ініціалізація $\gamma$, і коефіцієнти $\rho_t$ є простими. Ми ініціалізуємо GBM середнім значенням $\gamma = \frac{1}{n} \cdot \sum_{i = 1}^n y_i$ і покладемо всі коефіцієнти $\rho_t$ рівними 1.

Запустимо GBM і побудуємо два типи графіків: поточне наближення $\hat{f}(x)$ (синій графік) і кожне дерево $\hat{f}_t(x)$, побудоване на його псевдозалишках (зелений графік). Номер графіка відповідає номеру ітерації:

<img src='https://habrastorage.org/web/edb/328/98a/edb32898ad014d8d95782759d11f63fb.png' align='center' width='90%'>

На другій ітерації наші дерева вже відновили основну форму функції. Проте на першій ітерації видно, що алгоритм побудував лише «ліву гілку» функції ($x \in [-5, -4]$).

Це сталося тому, що дерева не мали достатньої глибини, щоб одразу побудувати симетричну гілку, і тому зосередилися на лівій гілці з більшою похибкою. Отже, права гілка з’явилася лише після другої ітерації.

Далі процес відбувається очікувано: на кожному кроці наші псевдозалишки зменшувалися, а GBM дедалі точніше наближав початкову функцію. Водночас дерева за своєю природою не можуть апроксимувати неперервну функцію, тому в цьому прикладі GBM не є ідеальним.

Щоб поекспериментувати з апроксимацією функцій за допомогою GBM, можна скористатися чудовою інтерактивною демонстрацією в блозі [Brilliantly wrong](http://arogozhnikov.github.io/2016/06/24/gradient_boosting_explained.html):

<img src='https://habrastorage.org/web/779/3e0/e66/7793e0e66b7d4871b6391a94cd5d4cf2.jpg' align='center' width='120%'>

<a id="3.-Loss-functions"></a>

# 3. Функції втрат

Якщо ми хочемо розв’язувати задачу класифікації замість регресії, що зміниться? Нам потрібно лише вибрати відповідну функцію втрат $L(y, f)$. Це найважливіший високорівневий елемент, який визначає, як саме відбуватиметься оптимізація і яких властивостей ми можемо очікувати від фінальної моделі.

Як правило, самостійно нічого вигадувати не потрібно – дослідники вже зробили це за нас. Сьогодні ми розглянемо функції втрат для двох найпоширеніших постановок: регресії $y \in \mathbb{R}$ і бінарної класифікації $y \in \left\{-1, 1\right\}$.

<a id="Regression-loss-functions"></a>

### Функції втрат для регресії

Почнімо із задачі регресії для $y \in \mathbb{R}$. Щоб вибрати відповідну функцію втрат, треба зрозуміти, яку саме властивість умовного розподілу $ (y|x)$ ми хочемо відновити. Найпоширеніші варіанти такі:

- $L_2$ loss або Gaussian loss:

$$
L(y, f) = (y - f)^2.
$$

Це класичне умовне середнє, тобто найпростіший і найуживаніший випадок. Якщо ми не маємо додаткової інформації або вимог щодо робастності моделі, можна використовувати Gaussian loss.

- $L_1$ loss або Laplacian loss:

$$
L(y, f) = |y - f|.
$$

На перший погляд ця функція не є диференційовною, однак фактично вона визначає умовну медіану. Як відомо, медіана є стійкою до викидів, тому в деяких випадках ця функція втрат є кращою. Штраф за великі відхилення тут менший, ніж у $L_2$.

- $L_q$ loss або Quantile loss:

$$
L(y, f) =
\begin{cases}
(1 - \alpha)\,|y - f|, & \text{якщо } y - f \le 0, \\
\alpha\,|y - f|, & \text{якщо } y - f > 0,
\end{cases}
\qquad \alpha \in (0, 1).
$$

Замість медіани ця функція використовує квантілі. Наприклад, $\alpha = 0.75$ відповідає 75%-квантилю. Бачимо, що ця функція є асиметричною і сильніше штрафує спостереження, розташовані праворуч від заданого квантиля.

<img src='https://habrastorage.org/web/6d5/e3a/09c/6d5e3a09c703491b947fde851e412ac0.png' align='center' width=100%>

Застосуймо функцію втрат $L_q$ до наших даних. Мета полягає у відновленні умовного 75%-квантиля косинуса. Зберімо всі складові для GBM:

- іграшкові дані $\left\{ (x_i, y_i) \right\}_{i=1, \ldots, 300}$;
- кількість ітерацій $M = 3$;
- функція втрат для квантилів:

    $$
    L_{0.75}(y, f) =
    \begin{cases}
    0.25\,|y - f|, & \text{якщо } y - f \le 0, \\
    0.75\,|y - f|, & \text{якщо } y - f > 0;
    \end{cases}
    $$


- градієнт $L_{0.75}(y, f)$ – це функція, зважена коефіцієнтом $\alpha = 0.75$. Ми навчатимемо дерево для задачі класифікації:

    $$
    r_i = -\left[\frac{\partial L\bigl(y_i, f(x_i)\bigr)}{\partial f(x_i)}\right]_{f(x)=\hat{f}(x)} = \alpha \, \mathbb{I}\!\left(y_i > \hat{f}(x_i)\right) - (1 - \alpha) \, \mathbb{I}\!\left(y_i \le \hat{f}(x_i)\right), \quad \text{для } i = 1, \ldots, 300;
    $$

- дерево рішень як базовий алгоритм $h(x)$;
- гіперпараметр дерев: $\mathrm{depth} = 2$.

Для початкового наближення візьмемо потрібний квантиль $y$. Проте ми нічого не знаємо про оптимальні коефіцієнти $\rho_t$, тому використаємо стандартний line search. Отримані результати такі:

<img src='https://habrastorage.org/web/0e6/7dd/614/0e67dd614076499e91c8c4238457ae4d.png' align='center' width='100%'>

Можна побачити, що на кожній ітерації$r_{i}$набуває лише двох можливих значень, але GBM усе одно здатен відновити початкову функцію.

Загальний результат GBM із квантильною функцією втрат збігається з результатом для квадратичної функції втрат, але зміщений на $\approx 0.135$. Водночас, якби ми використали 90%-квантиль, даних уже було б недостатньо, оскільки класи стали б незбалансованими. Це варто пам’ятати під час роботи з нетиповими задачами.

*"Кілька слів про функції втрат у регресії"*

Для задач регресії розроблено багато функцій втрат, деякі з них мають додаткові властивості. Наприклад, вони можуть бути робастними, як у [Huber loss function](https://en.wikipedia.org/wiki/Huber_loss). За невеликої кількості викидів ця функція поводиться як $L_2$, а після певного порога переходить до $L_1$. Це дає змогу зменшити вплив викидів і зосередитися на загальній картині.

Проілюструємо це на такому прикладі. Дані генеруються з функції $y = \frac{\sin(x)}{x}$ з доданим шумом, який є сумішшю нормального та Bernoulli-розподілів. На графіках A-D показано відповідні функції, а на F-H – відповідні моделі GBM (графік E відповідає початковій функції):

<img src='https://habrastorage.org/web/130/05b/222/13005b222e8a4eb68c3936216c05e276.jpg' align='center' width='150%'> 

[Оригінальний розмір](https://habrastorage.org/web/130/05b/222/13005b222e8a4eb68c3936216c05e276.jpg).

У цьому прикладі ми використали сплайни як базовий алгоритм. Отже, для бустингу зовсім не обов’язково використовувати лише дерева.

Можна чітко побачити різницю між функціями $L_2$, $L_1$ і Huber loss. Якщо підібрати оптимальні параметри для Huber loss, можна отримати найкраще можливе наближення серед усіх розглянутих варіантів. Відмінності також добре видно для 10%, 50% і 90%-квантилів.

На жаль, Huber loss function підтримують лише деякі популярні бібліотеки та пакети; наприклад, її підтримує h2o, але не XGBoost. Подібна ситуація стосується й ще екзотичніших речей, наприклад [умовних експектилів](https://www.slideshare.net/charthur/quantile-and-expectile-regression), однак знати про них усе одно корисно.

<a id="Classification-loss-functions"></a>

### Функції втрат для класифікації

Тепер розгляньмо задачу бінарної класифікації $y \in \left\{-1, 1\right\}$. Ми вже бачили, що GBM може оптимізувати навіть недиференційовні функції втрат. Технічно цю задачу можна розв’язувати і з регресійною функцією втрат $L_2$, але це було б некоректно.

Розподіл цільової змінної вимагає використання лог-правдоподібності, тому функції втрат мають бути задані для добутку цільової змінної на передбачення: $y \cdot f$. Найпоширеніші варіанти такі:

- Logistic loss або Bernoulli loss:

$$
L(y, f) = \log(1 + \exp(-2yf)).
$$

Вона має цікаву властивість: штрафує навіть правильно передбачені класи, що допомагає не лише мінімізувати втрати, а й сильніше розсовувати класи, навіть коли всі вони вже класифікуються правильно.

- AdaBoost loss:

$$
L(y, f) = \exp(-yf).
$$

Класичний AdaBoost еквівалентний GBM з цією функцією втрат. Концептуально вона дуже близька до logistic loss, але накладає сильніший експоненційний штраф за неправильне передбачення.

<img src='https://habrastorage.org/web/bf5/9de/dcf/bf59dedcfd9d49b18e89ce342b09ce69.png' align='center' width=100%>

Згенеруємо нові іграшкові дані для нашої задачі класифікації. За основу візьмемо наш зашумлений косинус, а для класів цільової змінної використаємо знакову функцію. Наші іграшкові дані мають такий вигляд (для наочності додано джитер-шум):

<img src='https://habrastorage.org/web/e72/513/78b/e7251378bf6d459ab1aeea7a1f1996a1.jpg' align='center' width='100%'>

Використаймо logistic loss, щоб побачити, що саме ми підсилюємо. Отже, знову зберімо всі складові, які нам потрібні для GBM:

- іграшкові дані $\left\{ (x_i, y_i) \right\}_{i=1, \ldots, 300}, \quad y_i \in \{-1, 1\}$;
- кількість ітерацій $M = 3$;
- Logistic loss як функція втрат; її градієнт обчислюється так:

    $$
    r_i = \frac{2 y_i}{1 + \exp\bigl(2 y_i \hat{f}(x_i)\bigr)}, \quad \text{для } i = 1, \ldots, 300;
    $$

- дерева рішень як базові алгоритми $h(x)$;
- гіперпараметри дерев рішень: глибина дерева дорівнює $2$.

Цього разу ініціалізація алгоритму є трохи складнішою. По-перше, наші класи незбалансовані (63% проти 37%). По-друге, не існує відомої аналітичної формули для ініціалізації цієї функції втрат, тому доводиться шукати $\hat{f}_0 = \gamma$ пошуком:

<img src='https://habrastorage.org/web/f8a/054/702/f8a05470271448d9bc0d4dc3e524a571.png' align='center' width=100%>

Наше оптимальне початкове наближення становить приблизно $-0.273$. Можна було здогадатися, що воно буде від’ємним, оскільки вигідніше передбачати все як найпоширеніший клас, але формули для точного значення тут немає. Тепер нарешті запустімо GBM і подивімося, що насправді відбувається всередині алгоритму:

<img src='https://habrastorage.org/web/7b4/ab0/5fa/7b4ab05fa0a543bfad94950e47f91568.png' align='center' width='100%'>

Алгоритм успішно відновив межу розділення між нашими класами. Можна побачити, як відокремлюються «нижні» області, адже дерева впевненіше роблять правильні передбачення для негативного класу, а також як формуються дві сходинки змішаних класів.

Очевидно, що ми маємо багато правильно класифікованих спостережень і певну кількість спостережень із великими похибками, які виникли через шум у даних.

<a id="Weights"></a>

### Ваги

Іноді трапляються ситуації, коли для нашої задачі потрібна більш спеціалізована функція втрат. Наприклад, у фінансових часових рядах ми можемо хотіти надавати більшу вагу значним рухам ряду; у задачі прогнозування відтоку клієнтів корисніше передбачати відтік клієнтів із великим LTV (lifetime value, тобто сумою коштів, яку клієнт принесе в майбутньому).

<img src='https://habrastorage.org/web/0c0/ad0/3a4/0c0ad03a4c4b46bfa5bcd5101678c9c4.jpg' align='center' width='90%'>

Статистик-ентузіаст міг би вигадати власну функцію втрат, виписати для неї градієнт (а для ефективнішого навчання – ще й гессіан) і ретельно перевірити, чи задовольняє ця функція потрібні властивості. Однак імовірність припуститися помилки, зіткнутися з обчислювальними труднощами і витратити непропорційно багато часу на дослідження тут дуже висока.

Натомість було придумано дуже простий інструмент (який на практиці згадують нечасто): зважування спостережень і призначення вагових функцій. Найпростіший приклад такого підходу – задання ваг для балансування класів. Загалом, якщо ми знаємо, що певна підмножина даних, як у вхідних змінних $x$, так і в цільовій змінній $y$, має для моделі більшу важливість, то просто призначаємо їй більшу вагу $w(x,y)$. Головна вимога полягає в дотриманні загальних умов для ваг:

$$
\begin{aligned}
w_i &\in \mathbb{R}, \\
w_i &\ge 0, \quad \text{для } i = 1, \ldots, n, \\
\sum_{i=1}^{n} w_i &> 0.
\end{aligned}
$$

Ваги можуть суттєво скоротити час, потрібний для підлаштування функції втрат під конкретну задачу, і водночас стимулюють експерименти з властивостями цільових моделей. Призначення цих ваг цілком залежить від творчого підходу. Ми просто додаємо скалярні ваги:

$$
\begin{aligned}
L_w(y, f) &= w \, L(y, f), \\
r_{it} &= -w_i \left[\frac{\partial L\bigl(y_i, f(x_i)\bigr)}{\partial f(x_i)}\right]_{f(x)=\hat{f}(x)}, \quad \text{для } i = 1, \ldots, n.
\end{aligned}
$$

Очевидно, що для довільних ваг ми не знаємо статистичних властивостей нашої моделі. Часто пов’язувати ваги зі значеннями $y$ буває надто складно. Наприклад, використання ваг, пропорційних $|y|$, у функції втрат $L_1$ не є еквівалентним $L_2$, оскільки градієнт не враховуватиме значення самих передбачень: $\hat{f}(x)$.

Усе це ми згадуємо для того, щоб краще розуміти власні можливості. Створімо для наших іграшкових даних досить екзотичні ваги. Визначимо сильно асиметричну вагову функцію такого вигляду:

$$
w(x) =
\begin{cases}
0.1, & \text{якщо } x \le 0, \\
0.1 + |\cos(x)|, & \text{якщо } x > 0.
\end{cases}
$$

<img src='https://habrastorage.org/web/8c2/1b1/aa4/8c21b1aa47134f7aa46b15ef910369b2.png' align='center' width='80%'>

Для таких ваг ми очікуємо отримати дві властивості: меншу деталізацію для від’ємних значень$x$і форму функції, близьку до початкового косинуса. Інші налаштування GBM беремо з попереднього прикладу класифікації, включно з line search для оптимальних коефіцієнтів. Подивімося, що вийшло:

<img src='https://habrastorage.org/web/afc/cca/72a/afccca72a0774990b685de37b0fe9d9f.png' align='center' width='100%'>

Ми отримали саме той результат, на який очікували. По-перше, добре видно, наскільки сильно відрізняються псевдозалишки; на початковій ітерації вони майже повторюють початковий косинус. По-друге, ліва частина графіка функції часто ігнорувалася на користь правої, оскільки там ваги були більшими.

По-третє, функція, яку ми отримали на третій ітерації, дістала достатньо уваги й почала нагадувати початковий косинус (щоправда, також почала трохи перенавчатися).

Ваги – це потужний, але ризикований інструмент, за допомогою якого можна керувати властивостями моделі. Якщо ви хочете оптимізувати власну функцію втрат, спершу доцільно спробувати розв’язати простішу задачу, але додати ваги до спостережень на власний розсуд.

<a id="4.-Conclusion"></a>

# 4. Висновки

Сьогодні ми розглянули теоретичні засади градієнтного бустингу. GBM – це не просто окремий алгоритм, а загальна методологія побудови ансамблів моделей. Крім того, ця методологія є достатньо гнучкою та розширюваною: можна навчати велику кількість моделей, враховуючи різні функції втрат і різноманітні схеми зважування.

Практика та ML-змагання показують, що в стандартних задачах (за винятком зображень, аудіо та дуже розріджених даних) GBM часто є найефективнішим алгоритмом, не кажучи вже про стекінг і високорівневі ансамблі, у яких GBM майже завжди присутній.

Крім того, існує багато адаптацій GBM [для навчання з підкріпленням](https://arxiv.org/abs/1603.04119) (Minecraft, ICML 2016). До речі, алгоритм Viola-Jones, який досі використовують у комп’ютерному баченні, [ґрунтується на AdaBoost](https://en.wikipedia.org/wiki/Viola%E2%80%93Jones_object_detection_framework#Learning_algorithm).

У цій лекції ми свідомо оминули питання регуляризації, стохастичності та гіперпараметрів GBM. Те, що впродовж усього викладу ми використовували малу кількість ітерацій $M = 3$, не було випадковістю. Якби ми взяли 30 дерев замість 3 і навчили GBM так, як описано вище, результат був би вже не таким передбачуваним:

<img src='https://raw.githubusercontent.com/radiukpavlo/intelligent-data-analysis/refs/heads/main/03_img/10_3_good_fit.png' align='center' width=100%>

<img src='https://raw.githubusercontent.com/radiukpavlo/intelligent-data-analysis/refs/heads/main/03_img/10_4_overfitting.png'   align='center' width=100%>

<img src='https://habrastorage.org/web/27f/0f5/3be/27f0f53be9424cb1afaffb9a0e32909f.jpg' align='center' width=100%>

[Інтерактивна демонстрація](http://arogozhnikov.github.io/2016/07/05/gradient_boosting_playground.html)

<a id="5-Useful-resources"></a>

# 5. Корисні ресурси

- [Оригінальна стаття](https://jerryfriedman.su.domains/ftp/trebst.pdf) про GBM від Джерома Фрідмана
- [Розділ у книзі The Elements of Statistical Learning](https://esl.hohoweiya.xyz/book/The%20Elements%20of%20Statistical%20Learning.pdf) від Hastie, Tibshirani, Friedman (сторінка 337)
- [Стаття у Wikipedia про градієнтний бустинг](https://en.wikipedia.org/wiki/Gradient_boosting)
- [Вступ до бустингових дерев (документація XGBoost)](https://xgboost.readthedocs.io/en/latest/tutorials/model.html)
- [Коли варто обирати CatBoost замість XGBoost або LightGBM: практичний посібник](https://towardsdatascience.com/catboost-vs-light-gbm-vs-xgboost-5f93620723db) на Neptune.ai
- [Порівняння та оптимізація алгоритмів Gradient Boosting Decision Tree](https://arxiv.org/abs/1809.04559), [XGBoost: масштабоване GPU-прискорене навчання](https://arxiv.org/abs/1806.11248) – порівняння CatBoost, LightGBM і XGBoost (без однозначного 100% переможця)